In [1]:
import numpy as np
from scipy import stats
import pandas as pd
import matplotlib.pyplot as plt

print("A/B test design notebook ready")

A/B test design notebook ready


In [2]:
# Business question:
# Does a 10% discount on low-repeat-purchase categories
# increase the repeat purchase rate?

print("=== A/B TEST DESIGN ===")
print("Test: Discount Offer Impact on Repeat Purchase Rate")
print("Control Group: No discount (existing customers, no offer)")
print("Treatment Group: 10% discount code sent to 'Needs Attention' RFM segment")
print("Success Metric: Repeat purchase rate within 30 days")

print("\nHypothesis Testing:")
print("Null Hypothesis (H0): Discount has no effect on repeat purchase rate")
print("Alternative Hypothesis (H1): Discount increases repeat purchase rate")

=== A/B TEST DESIGN ===
Test: Discount Offer Impact on Repeat Purchase Rate
Control Group: No discount (existing customers, no offer)
Treatment Group: 10% discount code sent to 'Needs Attention' RFM segment
Success Metric: Repeat purchase rate within 30 days

Hypothesis Testing:
Null Hypothesis (H0): Discount has no effect on repeat purchase rate
Alternative Hypothesis (H1): Discount increases repeat purchase rate


In [5]:
# Load data
df = pd.read_csv('../cleaned-data/master_orders.csv')
rfm = pd.read_csv('../cleaned-data/rfm_segments.csv')

# Since dataset contains mostly one order per customer,
# assume a business baseline repeat rate

baseline_repeat_rate = 0.15  # 15% assumed repeat rate

print(
    f"Baseline repeat purchase rate: "
    f"{baseline_repeat_rate:.3f} "
    f"({baseline_repeat_rate*100:.1f}%)"
)

print(
    "Note: Using assumed business baseline "
    "because customer repeat history is unavailable."
)

Baseline repeat purchase rate: 0.150 (15.0%)
Note: Using assumed business baseline because customer repeat history is unavailable.


In [6]:
# Parameters
p1 = baseline_repeat_rate       # Control conversion rate
p2 = baseline_repeat_rate + 0.05  # Expected lift (+5 percentage points)

alpha = 0.05   # 95% confidence level
power = 0.80   # 80% statistical power

from scipy.stats import norm

# Z-scores
z_alpha = norm.ppf(1 - alpha/2)
z_beta = norm.ppf(power)

# Pooled probability
p_pooled = (p1 + p2) / 2

# Sample size formula
n = (
    (
        z_alpha * np.sqrt(
            2 * p_pooled * (1 - p_pooled)
        )
        +
        z_beta * np.sqrt(
            p1 * (1 - p1)
            +
            p2 * (1 - p2)
        )
    )
    /
    (p2 - p1)
) ** 2

n = int(np.ceil(n))

print(
    f"Required sample size per group: "
    f"{n:,} customers"
)

print(
    f"Total customers needed: "
    f"{n*2:,}"
)

print(
    f"Available 'Needs Attention' customers: "
    f"{len(rfm[rfm['Segment'] == 'Needs Attention']):,}"
)

Required sample size per group: 906 customers
Total customers needed: 1,812
Available 'Needs Attention' customers: 16,678


In [7]:
# Split 'Needs Attention' customers randomly
needs_attention = (
    rfm[rfm['Segment'] == 'Needs Attention']
    ['customer_id']
    .tolist()
)

np.random.seed(42)

# Equal-size control and treatment groups
np.random.shuffle(needs_attention)

sample_size = min(
    n,
    len(needs_attention) // 2
)

control_ids = needs_attention[:sample_size]
treatment_ids = needs_attention[
    sample_size:sample_size*2
]

# Simulated repeat behavior
control_repeat = np.random.binomial(
    1,
    p1,
    len(control_ids)
)

treatment_repeat = np.random.binomial(
    1,
    p2,
    len(treatment_ids)
)

# Chi-square test
contingency = [
    [
        control_repeat.sum(),
        len(control_repeat)
        - control_repeat.sum()
    ],
    [
        treatment_repeat.sum(),
        len(treatment_repeat)
        - treatment_repeat.sum()
    ]
]

chi2, p_val, dof, expected = (
    stats.chi2_contingency(contingency)
)

print("\n=== SIMULATED A/B TEST RESULT ===")

print(
    f"Control repeat rate: "
    f"{control_repeat.mean():.3f}"
)

print(
    f"Treatment repeat rate: "
    f"{treatment_repeat.mean():.3f}"
)

print(f"Chi-Square = {chi2:.4f}")
print(f"P-value = {p_val:.4f}")

print(
    "Conclusion:",
    (
        "REJECT H0 — discount has a significant effect"
        if p_val < 0.05
        else "FAIL TO REJECT H0"
    )
)


=== SIMULATED A/B TEST RESULT ===
Control repeat rate: 0.162
Treatment repeat rate: 0.202
Chi-Square = 4.5387
P-value = 0.0331
Conclusion: REJECT H0 — discount has a significant effect


In [8]:
# If test succeeds: business impact

at_risk_customers = len(
    rfm[rfm['Segment'] == 'At-Risk']
)

needs_attention_customers = len(
    rfm[rfm['Segment'] == 'Needs Attention']
)

avg_order_value = df['payment_value'].mean()

lift = 0.05  # 5 percentage point improvement

additional_orders = (
    at_risk_customers
    + needs_attention_customers
) * lift

revenue_impact = (
    additional_orders
    * avg_order_value
)

campaign_cost = (
    at_risk_customers
    + needs_attention_customers
) * 5

print("\n=== BUSINESS IMPACT IF TEST SUCCEEDS ===")

print(
    f"Customers in test pool: "
    f"{at_risk_customers + needs_attention_customers:,}"
)

print(
    f"Expected additional orders: "
    f"{additional_orders:.0f}"
)

print(
    f"Estimated revenue uplift: "
    f"R${revenue_impact:,.2f}"
)

print(
    f"Campaign cost: "
    f"R${campaign_cost:,.2f}"
)

print(
    f"Net ROI: "
    f"{((revenue_impact - campaign_cost) / campaign_cost * 100):.0f}%"
)


=== BUSINESS IMPACT IF TEST SUCCEEDS ===
Customers in test pool: 39,924
Expected additional orders: 1996
Estimated revenue uplift: R$344,813.88
Campaign cost: R$199,620.00
Net ROI: 73%


## A/B Test Design Findings

### Objective

Evaluate whether a **10% discount offer** can improve the repeat purchase rate among disengaging customers in the **Needs Attention** segment.

### Experiment Design

* **Control Group:** Existing customers receiving no discount
* **Treatment Group:** Customers receiving a 10% discount code
* **Primary Success Metric:** Repeat purchase rate within 30 days
* **Hypothesis:**

  * **H0:** Discount has no effect on repeat purchase rate
  * **H1:** Discount increases repeat purchase rate

### Statistical Design

* **Baseline Repeat Purchase Rate:** 15.0%
* **Expected Lift:** +5 percentage points
* **Required Sample Size:** 906 customers per group
* **Available Target Customers:** 16,678 (*Needs Attention* segment)

### Simulated Test Result

* **Control Repeat Rate:** 16.2%
* **Treatment Repeat Rate:** 20.2%
* **P-value:** 0.0331

Since **p < 0.05**, the result is statistically significant. The discount intervention appears to positively influence repeat purchase behavior.

### Business Impact Estimation

* **Customers in Test Pool:** 39,924
* **Expected Additional Orders:** 1,996
* **Estimated Revenue Uplift:** R$344,813.88
* **Campaign Cost:** R$199,620.00
* **Estimated ROI:** 73%

### Recommendation

Launch a controlled pilot campaign targeting **Needs Attention** and **At-Risk** customers. If real-world results replicate the simulated uplift, the business can generate meaningful incremental revenue while maintaining a positive ROI.

### Limitation

This analysis is based on a simulated historical split because repeat purchase history in the cleaned dataset was limited. A live production experiment is recommended for validation.
